In [1]:
from typing import Any
import psycopg2
from loguru import logger
from psycopg2.extras import RealDictCursor

HEADERS = {
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
    "Accept-Encoding": "gzip, deflate, br, zstd",
    "Accept-Language": "en,es-ES;q=0.9,es;q=0.8",
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
}

PRODUCTION_DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "reactions_production_db",
    "user": "postgres",
    "password": "postgres",
}

def get_production_connection():
    """Connect to Phase B production database"""
    conn = psycopg2.connect(**PRODUCTION_DB_CONFIG)
    conn.set_client_encoding("UTF8")
    return conn

def search_by_template(template_id: str) -> dict[str, Any] | None:
    """
    Retrieve complete reaction information by reaction ID, including all related data
    from bonds and functional groups tables.

    Args:
        template_id (str): The unique reaction identifier (primary key)

    Returns:
        dict[str, Any]: dict containing:
            - All fields from reactions table
            - bonds_formed: list of bond labels
            - bonds_broken: list of bond labels
            - bonds_order_changed: list of bond labels
            - functional_groups_formed: list of functional group keys
            - functional_groups_broken: list of functional group keys
        Returns None if not found

    Example:
        >>> reaction = search_by_template(42)
        >>> if reaction:
        ...     print(reaction['retro_smarts_template'])
        ...     print(reaction['product_smiles'])
        ...     print(reaction['bonds_formed'])
        ...     print(reaction['functional_groups_formed'])
    """
    conn = get_production_connection()
    cursor = conn.cursor(cursor_factory=RealDictCursor)

    try:
        # Main query to get reaction data with all related information via JOINs
        cursor.execute(
            """
            SELECT
                r.reaction_id,
                r.template_hash,
                r.retro_smarts_template,
                r.canonical_smarts_template,
                r.mapped_rxn,
                r.product_smiles,
                r.reactant_smiles,
                r.dataset,
                r.source_row_id,
                r.staging_id,
                r.derive_version,
                r.created_at,
                
                -- Aggregate bonds formed
                COALESCE(
                    ARRAY_AGG(DISTINCT bf.bond_label) FILTER (WHERE bf.bond_label IS NOT NULL),
                    ARRAY[]::TEXT[]
                ) AS bonds_formed,
                
                -- Aggregate bonds broken
                COALESCE(
                    ARRAY_AGG(DISTINCT bb.bond_label) FILTER (WHERE bb.bond_label IS NOT NULL),
                    ARRAY[]::TEXT[]
                ) AS bonds_broken,
                
                -- Aggregate bonds order changed
                COALESCE(
                    ARRAY_AGG(DISTINCT boc.bond_label) FILTER (WHERE boc.bond_label IS NOT NULL),
                    ARRAY[]::TEXT[]
                ) AS bonds_order_changed,
                
                -- Aggregate functional groups formed
                COALESCE(
                    ARRAY_AGG(DISTINCT fgf.functional_group_key) FILTER (WHERE fgf.functional_group_key IS NOT NULL),
                    ARRAY[]::TEXT[]
                ) AS functional_groups_formed,
                
                -- Aggregate functional groups broken
                COALESCE(
                    ARRAY_AGG(DISTINCT fgb.functional_group_key) FILTER (WHERE fgb.functional_group_key IS NOT NULL),
                    ARRAY[]::TEXT[]
                ) AS functional_groups_broken
                
            FROM reactions r
            
            -- Join bonds formed
            LEFT JOIN reaction_bonds_formed rbf ON r.reaction_id = rbf.reaction_id
            LEFT JOIN bonds bf ON rbf.bond_id = bf.bond_id
            
            -- Join bonds broken
            LEFT JOIN reaction_bonds_broken rbb ON r.reaction_id = rbb.reaction_id
            LEFT JOIN bonds bb ON rbb.bond_id = bb.bond_id
            
            -- Join bonds order changed
            LEFT JOIN reaction_bonds_order_changed rboc ON r.reaction_id = rboc.reaction_id
            LEFT JOIN bonds boc ON rboc.bond_id = boc.bond_id
            
            -- Join functional groups formed
            LEFT JOIN reaction_functional_groups_formed rfgf ON r.reaction_id = rfgf.reaction_id
            LEFT JOIN functional_groups fgf ON rfgf.functional_group_id = fgf.functional_group_id
            
            -- Join functional groups broken
            LEFT JOIN reaction_functional_groups_broken rfgb ON r.reaction_id = rfgb.reaction_id
            LEFT JOIN functional_groups fgb ON rfgb.functional_group_id = fgb.functional_group_id
            
            WHERE r.reaction_id = %s
            
            GROUP BY
                r.reaction_id,
                r.template_hash,
                r.retro_smarts_template,
                r.canonical_smarts_template,
                r.mapped_rxn,
                r.product_smiles,
                r.reactant_smiles,
                r.dataset,
                r.source_row_id,
                r.staging_id,
                r.derive_version,
                r.created_at
        """,
            (template_id,),
        )

        result = cursor.fetchone()

        if result:
            logger.info(f"Found reaction with reaction_id: {template_id}")
            # Convert RealDictRow to regular dict and convert arrays to lists
            reaction_data = dict(result)
            
            # Convert PostgreSQL arrays to Python lists (if not already)
            for key in ['bonds_formed', 'bonds_broken', 'bonds_order_changed', 
                       'functional_groups_formed', 'functional_groups_broken']:
                if key in reaction_data and reaction_data[key] is None:
                    reaction_data[key] = []
            
            return reaction_data
        else:
            logger.warning(f"No reaction found with reaction_id: {template_id}")
            return None

    except Exception as e:
        logger.error(f"Error retrieving reaction by reaction_id '{template_id}': {e}")
        raise
    finally:
        cursor.close()
        conn.close()


In [2]:
_templates = []
TEMPLATES = [
    ["1914396"],
    ["1914397"],
    ["1914398"],
    ["1679759"],
    ["29648", "1914401", "1914414", "149046", "1914403"],
    ["1914405", "1914406", "1914407"],
    ["324328", "1914408", "733"],
    ["20810", "2895", "1914409", "1914410", "1914411", "74060"],
]
for template_ids in TEMPLATES:
    task_data = []
    for template_id in template_ids:
        data = search_by_template(template_id)
        if data is not None:
            task_data.append(data)
    _templates.append(task_data)



2025-10-22 08:36:15.014 | INFO     | __main__:search_by_template:148 - Found reaction with reaction_id: 1914396
2025-10-22 08:36:15.036 | INFO     | __main__:search_by_template:148 - Found reaction with reaction_id: 1914397
2025-10-22 08:36:15.046 | INFO     | __main__:search_by_template:148 - Found reaction with reaction_id: 1914398
2025-10-22 08:36:15.054 | INFO     | __main__:search_by_template:148 - Found reaction with reaction_id: 1679759
2025-10-22 08:36:15.063 | INFO     | __main__:search_by_template:148 - Found reaction with reaction_id: 29648
2025-10-22 08:36:15.073 | INFO     | __main__:search_by_template:148 - Found reaction with reaction_id: 1914401
2025-10-22 08:36:15.087 | INFO     | __main__:search_by_template:148 - Found reaction with reaction_id: 1914414
2025-10-22 08:36:15.100 | INFO     | __main__:search_by_template:148 - Found reaction with reaction_id: 149046
2025-10-22 08:36:15.107 | INFO     | __main__:search_by_template:148 - Found reaction with reaction_id: 191

In [3]:
print(_templates)

[[{'reaction_id': 1914396, 'template_hash': '05438ac597d4c98bf9e0b848e6c3f855f526356442ab6991a774e7b20e201138', 'retro_smarts_template': '[C:8]-[C@@H;D3;+0:9](-[C;H0;D3;+0:10](=[CH2;D1;+0:11])-[c:12])-[C@;H0;D4;+0:2](-[C:1])(-[OH;D1;+0:3])-[C:4](-[F;D1;H0:5])(-[F;D1;H0:6])-[F;D1;H0:7]>>[C:1]-[C;H0;D3;+0:2](=[O;H0;D1;+0:3])-[C:4](-[F;D1;H0:5])(-[F;D1;H0:6])-[F;D1;H0:7].[C:8]/[CH;D2;+0:9]=[C;H0;D3;+0:10](/[CH3;D1;+0:11])-[c:12]', 'canonical_smarts_template': '[C:1]-[C;H0;D3;+0:2](=[O;H0;D1;+0:3])-[C:4](-[F;D1;H0:5])(-[F;D1;H0:6])-[F;D1;H0:7].[C:8]/[CH;D2;+0:9]=[C;H0;D3;+0:10](/[CH3;D1;+0:11])-[c:12]>>[C:8]-[C@@H;D3;+0:9](-[C;H0;D3;+0:10](=[CH2;D1;+0:11])-[c:12])-[C@;H0;D4;+0:2](-[C:1])(-[OH;D1;+0:3])-[C:4](-[F;D1;H0:5])(-[F;D1;H0:6])-[F;D1;H0:7]', 'mapped_rxn': '[CH3:1]/[C:2]([c:3]1[cH:4][cH:5][cH:6][cH:7][cH:8]1)=[CH:9]/[CH2:10][CH2:11][CH2:12][C:13](=[O:14])[C:15]([F:16])([F:17])[F:18]>>[CH2:1]=[C:2]([c:3]1[cH:4][cH:5][cH:6][cH:7][cH:8]1)[C@@H:9]1[CH2:10][CH2:11][CH2:12][C@@:13]1([OH:1

In [6]:
import json
from datetime import datetime

def convert_dates(obj):
    """Recursively convert datetime objects to ISO format strings"""
    if isinstance(obj, datetime):
        return obj.isoformat()
    elif isinstance(obj, dict):
        return {key: convert_dates(value) for key, value in obj.items()}
    elif isinstance(obj, list):
        return [convert_dates(item) for item in obj]
    return obj

with open("templates.json", "w") as f:
    json.dump(convert_dates(_templates), f, indent=2)